<a href="https://colab.research.google.com/github/samnocera05-ui/mis433/blob/main/In_class_vibecoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [44]:
import requests
from google.colab import ai
from IPython.display import Image, display

In [45]:
breeds = requests.get("https://dog.ceo/api/breeds/list/all").json()["message"]
print("Main breeds:", len(breeds))
print("Selectable breeds:", sum(len(s) if s else 1 for s in breeds.values()))

Main breeds: 108
Selectable breeds: 165


In [46]:
# Option A: type a specific breed (leave blank to use preferences below)
breed_input = ""  #@param {type:"string"}

# Option B: adopter preferences
size = "Small"  #@param ["Any", "Small", "Medium", "Large"]
fur_color = "Golden"  #@param ["Any", "Black", "White", "Brown", "Golden", "Gray", "Spotted/Mixed"]
energy = "Low"  #@param ["Low", "Medium", "High"]
home = "House with yard"  #@param ["Apartment", "House with yard"]

budget = "Up to $250"  #@param ["Up to $100", "Up to $250", "Up to $500", "Over $500"]

all_breeds = requests.get("https://dog.ceo/api/breeds/list/all").json()["message"]
paths = [f"{b}/{s}" if s else b for b, subs in all_breeds.items() for s in (subs or [None])]

if breed_input.strip():
    # Typed breed: "golden retriever" -> "retriever/golden"
    words = breed_input.lower().strip().split()
    pick = f"{words[-1]}/{' '.join(words[:-1])}" if len(words) > 1 else words[0]
    reason_prompt = "In 2 sentences, describe the {breed} dog breed: size, temperament, and energy level."
else:
    # AI picks the best match from the real breed list
    pick = ai.generate_text(f"""From this list of dog breeds: {', '.join(paths)}
Pick the ONE breed that best fits: size {size}, fur color {fur_color}, energy level {energy}, living in a {home}.
Return only the breed exactly as written in the list, nothing else.""").strip().strip("`.").lower()
    reason_prompt = (f"In 2 sentences, explain why a {{breed}} fits someone wanting a {size.lower()}, "
                     f"{fur_color.lower()}-coated, {energy.lower()}-energy dog living in a {home.lower()}.")

if pick in paths:
    img_url = requests.get(f"https://dog.ceo/api/breed/{pick}/images/random").json()["message"]
    breed = " ".join(reversed(pick.split("/"))).title()
    print("Your match:", breed)
    display(Image(url=img_url, width=400))
    print("\nAbout this breed:\n", ai.generate_text(reason_prompt.format(breed=breed)))
else:
    breed = None
    name = breed_input.title() if breed_input.strip() else "dogs matching your preferences"
    print(f"Sorry, there are no {name} available for adoption right now.")

Your match: Pekinese



About this breed:
 The Pekingese is a compact toy breed that frequently features a luxurious golden coat and possesses a naturally calm, low-energy temperament ideal for indoor lounging. Because they require minimal exercise, a house with a yard provides more than enough safe space for their brief, leisurely strolls and gentle outdoor sunbathing.


In [48]:
from IPython.display import Markdown, HTML

if not breed:
    raise SystemExit("No dog available, so no profile was generated.")

def ask(prompt):
    return ai.generate_text(prompt).replace("```markdown", "").replace("```", "").strip()

context = f"Adoptable dog, breed: {breed}."

# Output 1: adoption profile
profile = ask(f"""{context} Write a shelter adoption profile in Markdown using exactly this format:
## 🐾 Meet [made-up name]!
**Breed:** {breed}
**Age:** [age]
**Personality:** [3 traits, comma separated]

[2 warm sentences about the dog]

**Ideal home:** [one sentence]
Return only the Markdown.""")

# Output 2: adopter's email to the shelter
email = ask(f"""{context} The adopter wants a {size.lower()}, {fur_color.lower()}-coated, {energy.lower()}-energy dog
for a {home.lower()}. Their adoption fee budget is {budget.lower()}.
Write a short, friendly email from the adopter to the shelter asking to meet this dog.
Politely mention the budget and ask what the adoption fee includes (e.g., vaccines, microchip, spay/neuter).
Format: 'Subject: ...' on the first line, then 5 sentences. Return plain text only.""")

# Split subject from body
lines = email.split("\n", 1)
subject = lines[0].replace("Subject:", "").strip()
body = lines[1].strip() if len(lines) > 1 else ""

# Display profile
display(Markdown(profile))

# Display email with dog photo
display(HTML(f"""
<div style="background:#fff;color:#222;border:1px solid #ddd;border-radius:12px;max-width:560px;font-family:sans-serif;margin-top:16px;overflow:hidden">
  <div style="background:#f1f3f4;padding:12px 16px;border-bottom:1px solid #ddd">
    <div><b>To:</b> adoptions@shelter.org</div>
    <div><b>Subject:</b> {subject}</div>
  </div>
  <div style="padding:16px">
    <div style="white-space:pre-wrap;line-height:1.5">{body}</div>
    <img src="{img_url}" style="width:100%;border-radius:8px;margin-top:14px">
    <div style="font-size:12px;color:#666;margin-top:4px">📎 Photo of the {breed} I'm interested in</div>
  </div>
</div>
"""))

# Plain-text version too
print("\n--- Plain text email ---\n")
print(email)

## 🐾 Meet Barnaby!
**Breed:** Pekinese
**Age:** 4 years old
**Personality:** Regal, affectionate, calm

Barnaby is a charming little lion-dog who loves nothing more than curling up on a plush blanket right beside his favorite person. With his soulful dark eyes and gentle nature, he brings a wonderful sense of peaceful companionship wherever he waddles.

**Ideal home:** Barnaby would thrive in a calm, quiet household ready to pamper him like the little prince he is.


--- Plain text email ---

Subject: Meeting your adoptable Pekingese

Hello, I saw your sweet golden Pekingese on your website and would love to schedule a time to come meet them! I live in a house with a fenced yard and am looking for a small, low-energy companion to join my home. My adoption fee budget is up to $250, so I wanted to politely confirm if that aligns with your shelter's pricing. Could you also share what the adoption fee covers, such as vaccinations, microchipping, and spay/neuter services? Thank you so much for your time, and I look forward to hearing from you!
